# Fine-tune with PyTorch

[Open in Colab](https://colab.research.google.com/github/Dsadd4/UltraMS/blob/main/cookbook/tutorials/pytorch_finetune.ipynb)

Use UltraMS as a standard `torch.nn.Module` in a training loop. The input file is the [five-spectrum DreaMS example](../../examples/data/NOTICE.md). Run cells from top to bottom.

## Install

In [ ]:
%pip install -q ultrams


## Read spectra

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch
from ultrams import UltraMS, read_spectra

sample = Path("examples/data/example_5_spectra.mgf")
if not sample.exists():
    sample = Path("example_5_spectra.mgf")
    if not sample.exists():
        urlretrieve(
            "https://raw.githubusercontent.com/Dsadd4/UltraMS/main/examples/data/example_5_spectra.mgf",
            sample,
        )
spectra = list(read_spectra(sample))
assert len(spectra) == 5
print(f"Loaded {len(spectra)} MS/MS spectra")

## Add demonstration labels

The five targets below are **placeholders made from file order**. They are not measured properties or research labels; replace them with your own labels for a scientific task.

In [ ]:
from torch.utils.data import DataLoader

records = [
    {**spectrum, "target": index / (len(spectra) - 1)}
    for index, spectrum in enumerate(spectra)
]
print([record["target"] for record in records])

## Train encoder and task head

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UltraMS.from_pretrained("unsupervised", device=device).train()
head = torch.nn.Linear(model.embedding_dim, 1).to(device)
loader = DataLoader(records, batch_size=2, collate_fn=model.batch_converter())
optimizer = torch.optim.AdamW([*model.parameters(), *head.parameters()], lr=1e-5)

losses = []
for batch in loader:
    batch = {name: value.to(device) for name, value in batch.items()}
    prediction = head(model(batch["peaks"], batch["attention_mask"], batch["precursor_mz"]))
    loss = torch.nn.functional.mse_loss(prediction.squeeze(-1), batch["target"])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
print("Batch losses (demonstration only):", losses)

## Save the run configuration and history

The weights trained on placeholder labels are only a code demonstration.

In [ ]:
import json

output = Path("ultrams_finetune_demo.json")
output.write_text(json.dumps({
    "model": "unsupervised",
    "target": "placeholder labels from file order",
    "batch_size": 2,
    "learning_rate": 1e-5,
    "loss_per_batch": losses,
}, indent=2) + "\n")
print(output.resolve())